In [7]:

import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import cv2, os
import shutil
from pathlib import Path
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import json

# Resolve project root (handles running from notebooks/ folder)
BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

# Paths
DATA_DIR = BASE_DIR / "data"
IMAGES_DIR = DATA_DIR / "images"
VIDEOS_DIR = DATA_DIR / "videos"
TEXT_DIR = DATA_DIR / "text"
print("✅ Setup complete")


✅ Setup complete


In [8]:

print("📄 Loading Text Datasets...")

# -------------------------
# 1️⃣ LIAR Dataset
# -------------------------
liar_ds = load_dataset("UKPLab/liar")["train"]
liar_df = pd.DataFrame(liar_ds)

# In LIAR:
# label_text column contains "true statement", "false statement", etc.
# We convert fake → 1, real → 0

liar_df['label'] = (liar_df['label_text'] != 'true statement').astype(int)

liar_df = liar_df[['text', 'label']]


# -------------------------
# 2️⃣ Fake News Dataset (fake.csv)
# -------------------------
fake_df = pd.read_csv(TEXT_DIR / "fake.csv")

fake_df = fake_df[['text']]
fake_df['label'] = 1   # Fake = 1


# -------------------------
# 3️⃣ True News Dataset (true.csv)
# -------------------------
true_df = pd.read_csv(TEXT_DIR / "true.csv")

true_df = true_df[['text']]
true_df['label'] = 0   # Real = 0


# -------------------------
# 4️⃣ Merge All Datasets
# -------------------------
text_df = pd.concat([liar_df, fake_df, true_df], ignore_index=True)

print(f"Total text samples: {len(text_df)}")


# -------------------------
# 5️⃣ Split 70 / 15 / 15
# -------------------------
train_text, temp = train_test_split(
    text_df,
    test_size=0.3,
    stratify=text_df['label'],
    random_state=42
)

val_text, test_text = train_test_split(
    temp,
    test_size=0.5,
    stratify=temp['label'],
    random_state=42
)


# -------------------------
# 6️⃣ Save Files
# -------------------------
train_text.to_csv(TEXT_DIR / "train.csv", index=False)
val_text.to_csv(TEXT_DIR / "val.csv", index=False)
test_text.to_csv(TEXT_DIR / "test.csv", index=False)

print("✅ Text splits saved")


📄 Loading Text Datasets...


Repo card metadata block was not found. Setting CardData to empty.


Total text samples: 55167
✅ Text splits saved


In [28]:

print("🖼️ Processing Images...")

# Assume images in data/images/celeb-df/, data/images/hf/
# Create train/val/test folders
for split in ['train', 'val', 'test']:
    for label in ['real', 'fake']:
        Path(f"data/images/{split}/{label}").mkdir(parents=True, exist_ok=True)

# Manual split example (adjust paths to your files)
image_paths = list(IMAGES_DIR.glob("**/*.jpg")) + list(IMAGES_DIR.glob("**/*.png"))
np.random.seed(42)
np.random.shuffle(image_paths)

# Split (assume labels from filename or metadata)
train_paths = image_paths[:5000]
val_paths = image_paths[5000:5600]
test_paths = image_paths[5600:]

# Copy files (pseudo-code - adjust label logic)
for paths, split in [(train_paths, 'train'), (val_paths, 'val'), (test_paths, 'test')]:
    for path in paths:
        label = 'fake' if 'fake' in str(path).lower() else 'real'
        shutil.copy(path, f"data/images/{split}/{label}/{path.name}")
print("✅ Images organized: 7K samples split")


🖼️ Processing Images...
✅ Images organized: 7K samples split


In [6]:

print("🎥 Extracting Video Frames...")

def infer_label_from_path(p: Path) -> str:
    parts = [x.lower() for x in p.parts]
    if any(x in ("fake", "celeb-fake") for x in parts):
        return "fake"
    if any(x in ("real", "celeb-real") for x in parts):
        return "real"
    return "real"  # default fallback

def extract_frames(video_path, output_dir, fps=8, max_frames=100):
    """Extract 5-10 FPS faces - for Xception+BiLSTM"""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print("⚠️ Cannot open:", video_path)
        return 0

    frames = []
    frame_count = 0

    while cap.isOpened() and frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_count % max(1, int(cap.get(cv2.CAP_PROP_FPS) / fps)) == 0:
            frame = cv2.resize(frame, (224, 224))
            frames.append(frame)
        frame_count += 1
    cap.release()

    video_id = video_path.stem
    label_dir = infer_label_from_path(video_path)

    video_dir = output_dir / video_id / label_dir
    video_dir.mkdir(parents=True, exist_ok=True)

    for i, frame in enumerate(frames):
        cv2.imwrite(str(video_dir / f"{i:04d}.jpg"), frame)
    return len(frames)

# Use absolute output path
frames_out = DATA_DIR / "videos" / "frames"
frames_out.mkdir(parents=True, exist_ok=True)

# Process a small sample first
video_files = list(VIDEOS_DIR.glob("**/*.mp4"))[:10]
for video in video_files:
    n = extract_frames(video, frames_out, fps=8)
    print(video.name, "frames:", n)

print("✅ Sample extraction done")


🎥 Extracting Video Frames...
id0_0000.mp4 frames: 34
id0_0001.mp4 frames: 34
id0_0002.mp4 frames: 34
id0_0003.mp4 frames: 34
id0_0004.mp4 frames: 34
id0_0005.mp4 frames: 34
id0_0006.mp4 frames: 34
id0_0007.mp4 frames: 34
id0_0008.mp4 frames: 34
id0_0009.mp4 frames: 34
✅ Sample extraction done
